# PraNet baseline reproduction — GitHub-connected Kaggle version

**Goal:** clone your own GitHub PraNet fork and reproduce the existing pretrained inference + evaluation pipeline. This notebook still uses the Kaggle datasets and evaluation script paths from the original notebook.

**Before running:** Edit `REPO_URL` in Step 1, attach your existing Kaggle datasets (weights, polyp segmentation, evaluation), enable **Internet** for GitHub cloning and **GPU** for inference. GitHub repository should be public for this unauthenticated example. `/kaggle/working` is temporary; GitHub is the permanent source of code.


In [ ]:
# STEP 1 — Clone YOUR PraNet GitHub repository into Kaggle's temporary workspace.
# Replace the URL below with your actual GitHub HTTPS clone URL.
# For this first test, use a PUBLIC repository. Private repositories need authentication.
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Muh-Ayyub/corruption-aware-segmentation.git"
REPO_DIR = Path("/kaggle/working/PraNet")

if "YOUR_USERNAME" in REPO_URL or "YOUR_REPOSITORY" in REPO_URL:
    raise ValueError("Edit REPO_URL to your GitHub repository HTTPS clone URL before running.")

if not (REPO_DIR / ".git").exists():
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
        raise RuntimeError(f"{REPO_DIR} exists but is not a Git clone. Remove/rename it first.")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    print("Repository already present; not pulling automatically to avoid overwriting experiments.")

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
print("Top-level files:", sorted(os.listdir(REPO_DIR)))
assert (REPO_DIR / "lib" / "PraNet_Res2Net.py").exists(), "PraNet architecture missing from cloned repository"


## Step 2 — Attach weights and polyp datasets
The following cells preserve the dataset paths and copying logic from your original notebook. If your Kaggle dataset paths change, update them here.

In [ ]:
import os, shutil
from pathlib import Path

print("--- Checkpoint files under /kaggle/input ---")
weights = sorted(Path('/kaggle/input').rglob('PraNet-19.pth'))
for path in weights:
    print(path)
if not weights:
    raise FileNotFoundError("PraNet-19.pth not found. Attach your pretrained-weights Kaggle dataset first.")
if len(weights) > 1:
    raise RuntimeError("Multiple PraNet-19.pth files found. Select the intended checkpoint explicitly.")
dst_dir = Path('/kaggle/working/PraNet/snapshots/PraNet-Res2Net50')
dst_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(weights[0], dst_dir / 'PraNet-19.pth')
print('Checkpoint ready:', dst_dir / 'PraNet-19.pth')


In [ ]:
import os

print("--- Searching inside /kaggle/input ---")
for root, dirs, files in os.walk('/kaggle/input'):
    if 'polyp' in root.lower() or 'segmentation' in root.lower():
        print(f"\nDirectory: {root}")
        print(f"Sub-directories: {dirs}")
        print(f"Files (first 10): {files[:10]}")

In [ ]:
import os, shutil

base_input = '/kaggle/input/datasets/muhammedayyub/polyp-segmentation'
target_data = '/kaggle/working/PraNet/data'

# 1. Clean destination and recreate structure
os.makedirs(f'{target_data}/TrainDataset', exist_ok=True)
os.makedirs(f'{target_data}/TestDataset', exist_ok=True)

# 2. Copy Training Data (Fixing folder name to 'images' if PraNet requires it)
src_train_imgs = f'{base_input}/TrainDataset/TrainDataset/image'
src_train_masks = f'{base_input}/TrainDataset/TrainDataset/masks'

dst_train_imgs = f'{target_data}/TrainDataset/images'
dst_train_masks = f'{target_data}/TrainDataset/masks'

if os.path.exists(src_train_imgs):
    shutil.copytree(src_train_imgs, dst_train_imgs, dirs_exist_ok=True)
    shutil.copytree(src_train_masks, dst_train_masks, dirs_exist_ok=True)
    print("✓ Successfully reorganized TrainDataset")

# 3. Copy Testing Data
src_test_dir = f'{base_input}/TestDataset/TestDataset'
dst_test_dir = f'{target_data}/TestDataset'

if os.path.exists(src_test_dir):
    for test_set in os.listdir(src_test_dir):
        src_path = os.path.join(src_test_dir, test_set)
        dst_path = os.path.join(dst_test_dir, test_set)
        if os.path.isdir(src_path):
            shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
    print("✓ Successfully reorganized TestDataset")

# Verify setup
print("\n--- Final Data Structure ---")
print("TrainDataset:", os.listdir(f'{target_data}/TrainDataset'))
print("TestDataset benchmarks:", os.listdir(f'{target_data}/TestDataset'))

## Step 3 — Kaggle inference compatibility

Your GitHub `lib/PraNet_Res2Net.py` already has `pretrained=False`, so the previous architecture-patching cells have been removed. The following cell **still overwrites `MyTest.py` in the temporary Kaggle clone** with the tested compatibility version from your original notebook (correct `lib` import and PNG saving). It does not change your GitHub repository. After this works, we can commit the compatibility fix to GitHub and remove this cell too.


In [ ]:
import os

test_file = '/kaggle/working/PraNet/MyTest.py'

# Corrected script matching PraNet's 'lib' directory structure
fixed_code = '''import torch
import torch.nn.functional as F
import numpy as np
import os, argparse
import imageio
from lib.PraNet_Res2Net import PraNet
from utils.dataloader import test_dataset

parser = argparse.ArgumentParser()
parser.add_argument('--testsize', type=int, default=352, help='testing size')
parser.add_argument('--pth_path', type=str, default='snapshots/PraNet-Res2Net50/PraNet-19.pth')
opt = parser.parse_args()

model = PraNet()
model.load_state_dict(torch.load(opt.pth_path))
model.cuda()
model.eval()

for dataset in ['CVC-ClinicDB', 'Kvasir', 'CVC-ColonDB', 'ETIS-LaribPolypDB', 'CVC-300']:
    data_path = './data/TestDataset/{}/'.format(dataset)
    save_path = './results/PraNet-Res2Net50/{}/'.format(dataset)
    os.makedirs(save_path, exist_ok=True)
    image_root = '{}/images/'.format(data_path)
    gt_root = '{}/masks/'.format(data_path)
    test_loader = test_dataset(image_root, gt_root, opt.testsize)
    
    for i in range(test_loader.size):
        image, gt, name = test_loader.load_data()
        gt = np.asarray(gt, np.float32)
        image = image.cuda()

        res, res2, res3, res4 = model(image)
        res = F.interpolate(res, size=gt.shape, mode='bilinear', align_corners=False)
        res = res.sigmoid().data.cpu().numpy().squeeze()
        
        # Normalize and cast float array to uint8 for Pillow/PNG compatibility
        res = (res - res.min()) / (res.max() - res.min() + 1e-8)
        res = (res * 255).astype(np.uint8)
        
        imageio.v2.imwrite(save_path + name, res)
    print('✓ Dataset finished:', dataset)
'''

with open(test_file, 'w') as f:
    f.write(fixed_code)

print("✓ Fixed import in MyTest.py (from lib.PraNet_Res2Net import PraNet).")

## Step 4 — Run baseline inference

In [ ]:
%cd /kaggle/working/PraNet
!python MyTest.py --testsize 352 --pth_path snapshots/PraNet-Res2Net50/PraNet-19.pth


## Step 5 — Evaluate predictions
Uses the same evaluation dataset location and `pysodmetrics` package as your original notebook.

In [ ]:
!cp /kaggle/input/datasets/muhammedayyub/evaluation/eval_pranet.py /kaggle/working/PraNet

In [ ]:
%cd /kaggle/working/PraNet
!pip install pysodmetrics -q

In [ ]:
!python eval_pranet.py